In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATH
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Demographics"
)

xpt_file = DATA_DIR / "DEMO_C.xpt"
csv_file = DATA_DIR / "DEMO_C_corrected.csv"

# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

print("=" * 70)
print("NHANES 2003-2004 DEMO_C XPT -> CSV")
print("=" * 70)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. KNOWN PANDAS XPT ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(5.397605346934028e-79)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


# ============================================================
# 4. CHECK ORIGINAL FILE BEFORE CHANGING ANYTHING
# ============================================================

zero_counts_before = (
    df[numeric_cols]
    .eq(0)
    .sum()
)

tiny_counts_before = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

total_zeros_before = int(
    zero_counts_before.sum()
)

total_tiny_before = int(
    tiny_counts_before.sum()
)

tiny_columns_before = int(
    (tiny_counts_before > 0).sum()
)


print("\n--- BEFORE CORRECTION ---")

print(
    "True numeric zeros:",
    f"{total_zeros_before:,}"
)

print(
    f"Tiny value {TINY_VALUE:.16e}: "
    f"{total_tiny_before:,} occurrences "
    f"across {tiny_columns_before:,} columns"
)


if total_tiny_before > 0:

    print("\nVariables containing tiny value:")

    tiny_report = pd.DataFrame({
        "tiny_count":
            tiny_counts_before[tiny_counts_before > 0],

        "true_zero_count":
            zero_counts_before[tiny_counts_before > 0]
    })

    tiny_report = tiny_report.sort_values(
        "tiny_count",
        ascending=False
    )

    print(tiny_report)


# ============================================================
# 5. RESTORE THE KNOWN ZERO ARTIFACT
# ============================================================

for col in numeric_cols:

    mask = df[col].eq(TINY_VALUE)

    if mask.any():

        df.loc[mask, col] = 0.0


# ============================================================
# 6. DEMO VARIABLE TYPE RULES
# ============================================================

#
# These are genuine continuous decimal variables.
# DO NOT convert these to integers.
#

CONTINUOUS_COLUMNS = {
    "INDFMPIR",   # Poverty Income Ratio
    "WTINT2YR",   # Interview sample weight
    "WTMEC2YR"    # MEC examination sample weight
}


# All other numeric DEMO variables are codes,
# integer ages/counts, identifiers, PSU/stratum values, etc.
#
# Examples:
#
# SEQN      21005.0 -> 21005
# RIAGENDR      1.0 -> 1
# RIDAGEYR     19.0 -> 19
# DMDEDUC3      0.0 -> 0
#
# Missing values remain blank in CSV.

integer_columns = [
    col
    for col in numeric_cols
    if col not in CONTINUOUS_COLUMNS
]


# ============================================================
# 7. SAFETY CHECK BEFORE INTEGER CONVERSION
# ============================================================

print("\n--- INTEGER FIDELITY CHECK ---")

for col in integer_columns:

    values = df[col].dropna()

    if values.empty:
        continue

    non_integer = ~np.isclose(
        values,
        np.round(values)
    )

    if non_integer.any():

        bad_values = values[non_integer]

        raise ValueError(
            f"{col} contains unexpected decimal values: "
            f"{bad_values.head().tolist()}"
        )


print(
    "PASS: All code/count/identifier variables "
    "contain whole-number values."
)


# ============================================================
# 8. CONVERT CODE/INTEGER VARIABLES
# ============================================================

for col in integer_columns:

    df[col] = (
        df[col]
        .round()
        .astype("Int64")
    )


# ============================================================
# 9. IMPORTANT DEMO ZERO CHECKS
# ============================================================

print("\n--- ZERO CHECK AFTER CORRECTION ---")

variables_to_check = [
    "RIDAGEYR",
    "RIDAGEMN",
    "RIDAGEEX",
    "DMDEDUC3",
    "INDFMPIR",
    "WTMEC2YR"
]

for col in variables_to_check:

    if col in df.columns:

        zero_count = int(
            df[col].eq(0).sum()
        )

        print(
            f"{col:<10} zeros = "
            f"{zero_count:,}"
        )


# ============================================================
# 10. CDC SPECIFIC VALIDATION
# ============================================================

print("\n--- CDC CODEBOOK VALIDATION ---")


# DEMO_C should contain 10,122 participants

if len(df) == 10122:
    print("PASS: Row count = 10,122")
else:
    print(
        "FAIL: Unexpected row count:",
        len(df)
    )


# ------------------------------------------------------------
# SDDSRVYR
# CDC: code 3 = NHANES 2003-2004, count 10,122
# ------------------------------------------------------------

if "SDDSRVYR" in df.columns:

    expected = {
        3: 10122
    }

    actual = (
        df["SDDSRVYR"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    print(
        "SDDSRVYR:",
        "PASS" if actual == expected else "CHECK",
        actual
    )


# ------------------------------------------------------------
# RIDSTATR
#
# CDC:
# 1 = Interviewed only                  479
# 2 = Interviewed + MEC examined      9643
# ------------------------------------------------------------

if "RIDSTATR" in df.columns:

    expected = {
        1: 479,
        2: 9643
    }

    actual = (
        df["RIDSTATR"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    print(
        "RIDSTATR:",
        "PASS" if actual == expected else "CHECK",
        actual
    )


# ------------------------------------------------------------
# RIAGENDR
#
# CDC:
# 1 = Male      4970
# 2 = Female    5152
# ------------------------------------------------------------

if "RIAGENDR" in df.columns:

    expected = {
        1: 4970,
        2: 5152
    }

    actual = (
        df["RIAGENDR"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    print(
        "RIAGENDR:",
        "PASS" if actual == expected else "CHECK",
        actual
    )


# ------------------------------------------------------------
# DMDEDUC3
#
# CDC explicitly:
# 0 = Never attended / Kindergarten only
# Count = 205
# ------------------------------------------------------------

if "DMDEDUC3" in df.columns:

    educ_zero = int(
        df["DMDEDUC3"].eq(0).sum()
    )

    if educ_zero == 205:

        print(
            "PASS: DMDEDUC3 code 0 = 205 "
            "(matches CDC)"
        )

    else:

        print(
            "FAIL: DMDEDUC3 zero count =",
            educ_zero,
            "expected 205"
        )


# ------------------------------------------------------------
# WTMEC2YR
#
# Interview-only participants do not receive MEC exam weight.
# The attached XPT has 479 zero MEC weights after correction.
# Compare directly with RIDSTATR=1.
# ------------------------------------------------------------

if (
    "WTMEC2YR" in df.columns
    and "RIDSTATR" in df.columns
):

    mec_zero = int(
        df["WTMEC2YR"].eq(0).sum()
    )

    interview_only = int(
        df["RIDSTATR"].eq(1).sum()
    )

    print(
        "WTMEC2YR zero count:",
        mec_zero
    )

    if mec_zero == interview_only:

        print(
            "PASS: WTMEC2YR zeros match "
            "interview-only participants:",
            interview_only
        )

    else:

        print(
            "CHECK: WTMEC2YR zeros do not match "
            "RIDSTATR interviewed-only count."
        )


# ============================================================
# 11. SEQN FIDELITY
# ============================================================

print("\n--- SEQN VALIDATION ---")

if "SEQN" in df.columns:

    print(
        "dtype:",
        df["SEQN"].dtype
    )

    print(
        "Missing:",
        int(df["SEQN"].isna().sum())
    )

    print(
        "Duplicates:",
        int(df["SEQN"].duplicated().sum())
    )

    print(
        "Minimum:",
        df["SEQN"].min()
    )

    print(
        "Maximum:",
        df["SEQN"].max()
    )

    if (
        df["SEQN"].isna().sum() == 0
        and df["SEQN"].duplicated().sum() == 0
    ):

        print(
            "PASS: SEQN is unique and non-missing."
        )


# ============================================================
# 12. CHECK CONTINUOUS VARIABLES
# ============================================================

print("\n--- CONTINUOUS VARIABLE CHECK ---")

for col in CONTINUOUS_COLUMNS:

    if col in df.columns:

        print(
            f"{col:<10}",
            "dtype =", df[col].dtype,
            "min =", df[col].min(),
            "max =", df[col].max(),
            "missing =", int(df[col].isna().sum())
        )


# ============================================================
# 13. FINAL TINY-VALUE CHECK
# ============================================================

remaining_tiny = int(
    df.select_dtypes(include=[np.number])
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print("\n--- FINAL ZERO FIDELITY ---")

if remaining_tiny == 0:

    print(
        "PASS: No 5.397605346934028e-79 "
        "values remain."
    )

else:

    print(
        "FAIL:",
        remaining_tiny,
        "tiny values remain."
    )


# ============================================================
# 14. EXPORT CSV
# ============================================================

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 15. RE-READ CSV FOR ROUND-TRIP CHECK
# ============================================================

check_df = pd.read_csv(
    csv_file,
    low_memory=False
)


print("\n--- CSV ROUND-TRIP CHECK ---")


# Check SEQN does not contain .0 in actual CSV text

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    header = f.readline()

    first_data_lines = [
        f.readline()
        for _ in range(10)
    ]


seqn_has_decimal = any(
    line.split(",")[0].endswith(".0")
    for line in first_data_lines
)


if not seqn_has_decimal:

    print(
        "PASS: SEQN is exported without .0"
    )

else:

    print(
        "FAIL: SEQN still contains .0"
    )


# Final tiny value check after reading CSV

csv_numeric = check_df.select_dtypes(
    include=[np.number]
)

tiny_after_csv = int(
    csv_numeric
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


if tiny_after_csv == 0:

    print(
        "PASS: Tiny zero artifact "
        "did not return after CSV export."
    )

else:

    print(
        "FAIL:",
        tiny_after_csv,
        "tiny values found after CSV export."
    )


print("\n" + "=" * 70)
print("DEMO_C CONVERSION COMPLETE")
print("=" * 70)

NHANES 2003-2004 DEMO_C XPT -> CSV
Rows: 10,122
Columns: 44

--- BEFORE CORRECTION ---
True numeric zeros: 0
Tiny value 5.3976053469340279e-79: 1,410 occurrences across 6 columns

Variables containing tiny value:
          tiny_count  true_zero_count
WTMEC2YR         479                0
RIDAGEYR         477                0
DMDEDUC3         205                0
INDFMPIR         177                0
RIDAGEMN          53                0
RIDAGEEX          19                0

--- INTEGER FIDELITY CHECK ---
PASS: All code/count/identifier variables contain whole-number values.

--- ZERO CHECK AFTER CORRECTION ---
RIDAGEYR   zeros = 477
RIDAGEMN   zeros = 53
RIDAGEEX   zeros = 19
DMDEDUC3   zeros = 205
INDFMPIR   zeros = 177
WTMEC2YR   zeros = 479

--- CDC CODEBOOK VALIDATION ---
PASS: Row count = 10,122
SDDSRVYR: PASS {np.int64(3): 10122}
RIDSTATR: PASS {np.int64(1): 479, np.int64(2): 9643}
RIAGENDR: PASS {np.int64(1): 4970, np.int64(2): 5152}
PASS: DMDEDUC3 code 0 = 205 (matches CDC)
WT